# Example 02: Streaming（流式输出）

SSE 逐 chunk 解析 + 思考 token 与答案 token 分离显示

**Demonstrates:**
- Enabling streaming (`stream=True`)
- Parsing SSE chunks as they arrive (delta content)
- Collecting the full response from chunks
- Accessing usage stats from the final chunk (`stream_options`)
- Separating `reasoning_content` from answer `content`

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server first (see quickstart.md)
```

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
)

MODEL = os.environ.get("HY3_MODEL", "hy3")

## Basic Streaming（基础流式）

逐 token 打印，最后从 usage chunk 读取 token 统计。

In [ ]:
prompt = "List 3 benefits of functional programming. Be concise."
print(f"Prompt: {prompt}")
print("Response: ", end="", flush=True)

full_content = ""
usage = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.9,
    top_p=1.0,
    stream=True,
    stream_options={"include_usage": True},  # request token count in final chunk
    extra_body={"chat_template_kwargs": {"reasoning_effort": "no_think"}},
)

for chunk in stream:
    if chunk.usage is not None:          # final chunk carries usage stats
        usage = chunk.usage
        continue
    if not chunk.choices:
        continue

    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)
        full_content += delta.content

    if chunk.choices[0].finish_reason == "stop":
        break

print()  # newline
if usage:
    print(f"Tokens — prompt: {usage.prompt_tokens}, completion: {usage.completion_tokens}, total: {usage.total_tokens}")

**Sample output:**
```
Prompt: List 3 benefits of functional programming. Be concise.
Response: 1. **Immutability** – Avoids shared state, reducing bugs.
2. **Composability** – Small, pure functions combine easily.
3. **Testability** – Pure functions are trivial to unit-test.
Tokens — prompt: 18, completion: 47, total: 65
```

## Streaming with Reasoning（流式 + 思考 token 分离）

使用 `reasoning_effort="high"` 时，vLLM 在 `delta.reasoning_content` 中返回思考过程，
答案内容在 `delta.content` 中——两者分开显示。

In [ ]:
prompt = "What is 17 × 23? Show your work."
print(f"Prompt: {prompt}")
print("[Thinking] ", end="", flush=True)

thinking_content = ""
answer_content = ""
in_think = True

stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.9,
    top_p=1.0,
    stream=True,
    stream_options={"include_usage": True},
    extra_body={"chat_template_kwargs": {"reasoning_effort": "high"}},
)

for chunk in stream:
    if chunk.usage is not None:
        continue
    if not chunk.choices:
        continue

    delta = chunk.choices[0].delta

    # Check reasoning_content by identity (not truthiness) so empty-string tokens
    # are not silently dropped, and reasoning after answer-start is still captured.
    reasoning_delta = getattr(delta, "reasoning_content", None)
    if reasoning_delta is not None and reasoning_delta != "":
        print(reasoning_delta, end="", flush=True)
        thinking_content += reasoning_delta

    if delta.content:
        if in_think:
            print("\n[Answer] ", end="", flush=True)
            in_think = False
        print(delta.content, end="", flush=True)
        answer_content += delta.content

print()

**Sample output:**
```
Prompt: What is 17 × 23? Show your work.
[Thinking] I need to compute 17 × 23. I can break this down:
17 × 23 = 17 × 20 + 17 × 3 = 340 + 51 = 391
[Answer] 17 × 23 = **391**

Work:
- 17 × 20 = 340
- 17 × 3  = 51
- 340 + 51 = **391**
```